# Virginia Piedmont native phenology

This notebook analyzes seasonal prevalence of native Lepidoptera and Anthophila in the Virginia Piedmont using iNaturalist's aggregate histogram endpoints rather than paginated raw observation fetches.

**Approach:**
- `/observations/histogram` (week_of_year and month_of_year) returns complete all-time counts in a single API call per query — no pagination, no truncation.
- `native=true` delegates nativity filtering to iNat's own establishment-means index; for county-level place IDs, iNat resolves nativity up the ancestor hierarchy to Virginia state level.
- Effort normalization divides each group's weekly count by total verifiable observations in the same week, correcting for observer seasonality.
- Per-taxon detail uses `species_counts` (also unpaginated) to identify top native taxa, then one monthly histogram call per taxon.

**Total API calls: ~24** (vs ~175+ for the prior paginated approach, which captured ≤3% of Lepidoptera observations).

**Outputs:**
- Weekly overview chart: raw counts + effort-normalized fraction, all focus groups and life stage slices
- Monthly outreach chart: top-N native taxa per group, bar chart suitable for poster/web use
- Annotation coverage report: flags sparse life stage slices

In [1]:
from pathlib import Path
import sys

import pandas as pd
import plotly.express as px

repo_root = Path.cwd()
if not (repo_root / 'helpers.py').exists() and (repo_root.parent.parent / 'helpers.py').exists():
    repo_root = repo_root.parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from helpers import get_inat_session

# 27 Piedmont counties + 9 independent cities (iNaturalist place IDs)
PIEDMONT_PLACE_IDS = [
    # Counties
    2913,   # Albemarle
    1660,   # Amelia
    1372,   # Appomattox
    1719,   # Arlington
    733,    # Brunswick
    2589,   # Buckingham
    617,    # Campbell
    1662,   # Charlotte
    2917,   # Culpeper
    2590,   # Cumberland
    738,    # Fairfax
    1494,   # Fauquier
    1721,   # Fluvanna
    2920,   # Goochland
    1075,   # Halifax
    1186,   # Henry
    3032,   # Louisa
    739,    # Loudoun
    1724,   # Lunenburg
    1493,   # Mecklenburg
    742,    # Nottoway
    743,    # Orange
    1664,   # Pittsylvania
    1491,   # Powhatan
    2923,   # Prince Edward
    744,    # Prince William
    2925,   # Spotsylvania
    # Independent cities
    13446,  # Alexandria
    13435,  # Charlottesville
    13434,  # Danville
    108238, # Fairfax (city)
    13451,  # Falls Church
    13483,  # Lynchburg
    13410,  # Manassas
    13430,  # Manassas Park
    13445,  # Martinsville
]

LEPIDOPTERA_TAXON_ID = 47157
ANTHOPHILA_TAXON_ID  = 630955
LIFESTAGE_TERM_ID    = 1
LIFESTAGE_ADULT      = 2
LIFESTAGE_PUPA       = 4
LIFESTAGE_LARVA      = 6
TOP_N                = 5   # top taxa per focus group for the outreach chart

HISTOGRAM_URL    = 'https://api.inaturalist.org/v1/observations/histogram'
SPECIES_COUNTS_URL = 'https://api.inaturalist.org/v1/observations/species_counts'

CACHE_PATH         = repo_root / 'notebooks' / 'exploratory' / 'cache' / 'va_piedmont_phenology'
CACHE_MAX_AGE_DAYS = 7

session = get_inat_session()

In [2]:
# -- Preflight: total observation counts (informational) --
_place_str = ','.join(str(x) for x in PIEDMONT_PLACE_IDS)

for _label, _taxon_id in [('Lepidoptera', LEPIDOPTERA_TAXON_ID), ('Anthophila', ANTHOPHILA_TAXON_ID)]:
    _r = session.get(
        'https://api.inaturalist.org/v1/observations',
        params={'place_id': _place_str, 'taxon_id': _taxon_id, 'verifiable': 'true', 'per_page': 0},
    )
    _total = _r.json().get('total_results', '?')
    print(f'{_label}: {_total:,} total verifiable observations across Piedmont places (all will be captured)')

Lepidoptera: 210,144 total verifiable observations across Piedmont places (all will be captured)
Anthophila: 36,534 total verifiable observations across Piedmont places (all will be captured)


In [3]:
# -- Fetch group histograms (week_of_year + month_of_year) and effort denominators --
# Checks disk cache first; re-fetches if absent or stale.

_query_specs = [
    {'focus_group': 'Lepidoptera', 'life_stage_bucket': 'overall',
     'extra': {'taxon_id': LEPIDOPTERA_TAXON_ID}},
    {'focus_group': 'Lepidoptera', 'life_stage_bucket': 'larva',
     'extra': {'taxon_id': LEPIDOPTERA_TAXON_ID, 'term_id': LIFESTAGE_TERM_ID, 'term_value_id': LIFESTAGE_LARVA}},
    {'focus_group': 'Lepidoptera', 'life_stage_bucket': 'pupa',
     'extra': {'taxon_id': LEPIDOPTERA_TAXON_ID, 'term_id': LIFESTAGE_TERM_ID, 'term_value_id': LIFESTAGE_PUPA}},
    {'focus_group': 'Lepidoptera', 'life_stage_bucket': 'adult',
     'extra': {'taxon_id': LEPIDOPTERA_TAXON_ID, 'term_id': LIFESTAGE_TERM_ID, 'term_value_id': LIFESTAGE_ADULT}},
    {'focus_group': 'Anthophila', 'life_stage_bucket': 'overall',
     'extra': {'taxon_id': ANTHOPHILA_TAXON_ID}},
]

_base = {'place_id': _place_str, 'verifiable': 'true'}
_cache_files = [CACHE_PATH / 'hist_df.parquet', CACHE_PATH / 'effort_df.parquet']
_need_fetch = True

if all(f.exists() for f in _cache_files):
    _age_days = (pd.Timestamp.now().timestamp() - _cache_files[0].stat().st_mtime) / 86400
    if _age_days < CACHE_MAX_AGE_DAYS:
        hist_df   = pd.read_parquet(_cache_files[0])
        effort_df = pd.read_parquet(_cache_files[1])
        print(f'Loaded from cache ({_age_days:.1f}d old). Delete {CACHE_PATH} to force refresh.')
        _need_fetch = False

if _need_fetch:
    _hist_rows, _effort_rows, _api_calls = [], [], 0

    for spec in _query_specs:
        for interval in ('week_of_year', 'month_of_year'):
            # native=true: iNat resolves nativity against each observation's place and its
            # ancestors, so county-level place IDs effectively use VA state-level nativity data.
            params = {**_base, **spec['extra'], 'interval': interval, 'native': 'true'}
            r = session.get(HISTOGRAM_URL, params=params)
            r.raise_for_status()
            _api_calls += 1
            for key, count in r.json().get('results', {}).get(interval, {}).items():
                _hist_rows.append({
                    'focus_group':       spec['focus_group'],
                    'life_stage_bucket': spec['life_stage_bucket'],
                    'interval':          interval,
                    'bin':               int(key),
                    'count':             int(count),
                })

    # Effort denominators: all verifiable obs in Piedmont, no nativity filter
    for interval in ('week_of_year', 'month_of_year'):
        params = {**_base, 'interval': interval}
        r = session.get(HISTOGRAM_URL, params=params)
        r.raise_for_status()
        _api_calls += 1
        for key, count in r.json().get('results', {}).get(interval, {}).items():
            _effort_rows.append({'interval': interval, 'bin': int(key), 'effort': int(count)})

    hist_df   = pd.DataFrame(_hist_rows)
    effort_df = pd.DataFrame(_effort_rows)

    hist_df = hist_df.merge(effort_df, on=['interval', 'bin'], how='left')
    hist_df['fraction'] = hist_df['count'] / hist_df['effort'].replace(0, float('nan'))

    CACHE_PATH.mkdir(parents=True, exist_ok=True)
    hist_df.to_parquet(_cache_files[0], index=False)
    effort_df.to_parquet(_cache_files[1], index=False)
    print(f'Fetched with {_api_calls} API calls. Cached to {CACHE_PATH}.')

# Verification: histogram sums should be close to preflight total_results
print()
for fg, taxon_id in [('Lepidoptera', LEPIDOPTERA_TAXON_ID), ('Anthophila', ANTHOPHILA_TAXON_ID)]:
    _sum = hist_df[
        (hist_df['focus_group'] == fg) &
        (hist_df['life_stage_bucket'] == 'overall') &
        (hist_df['interval'] == 'week_of_year')
    ]['count'].sum()
    print(f'{fg} overall (weekly histogram sum): {_sum:,} native obs')

Fetched with 12 API calls. Cached to c:\Users\drsvs\Desktop\code\pynat\notebooks\exploratory\cache\va_piedmont_phenology.

Lepidoptera overall (weekly histogram sum): 164,120 native obs
Anthophila overall (weekly histogram sum): 20,594 native obs


In [4]:
# -- Verification: spot-check known species against expected phenology --
# These are biological ground-truth checks. If they fail, the native=true
# filter or place ID set is behaving unexpectedly and the charts below
# should not be trusted for outreach.
#
# Expected patterns (VA Piedmont):
#   Danaus plexippus   (Monarch, taxon 48662)    — bimodal: Apr-May AND Sep-Oct
#   Papilio glaucus    (Tiger Swallowtail, 53827) — peak May-Jun, secondary Aug
#   Apis mellifera     (Honeybee, 47219)          — should be ABSENT (introduced)

_spot_checks = [
    {'taxon_id': 48662,  'common_name': 'Monarch',              'expect': 'native',     'peak_months': {4, 5, 9, 10}},
    {'taxon_id': 53827,  'common_name': 'Eastern Tiger Swallowtail', 'expect': 'native', 'peak_months': {5, 6, 8}},
    {'taxon_id': 47219,  'common_name': 'European Honeybee',    'expect': 'absent',     'peak_months': set()},
]

print('Spot-check results:')
print()
for sc in _spot_checks:
    # species_counts with native=true — should find native taxa, not introduced
    r_native = session.get(
        SPECIES_COUNTS_URL,
        params={**_base, 'taxon_id': sc['taxon_id'], 'native': 'true', 'per_page': 1},
    )
    r_native.raise_for_status()
    native_count = r_native.json().get('total_results', 0)

    r_introduced = session.get(
        SPECIES_COUNTS_URL,
        params={**_base, 'taxon_id': sc['taxon_id'], 'introduced': 'true', 'per_page': 1},
    )
    r_introduced.raise_for_status()
    introduced_count = r_introduced.json().get('total_results', 0)

    if sc['expect'] == 'absent':
        status = '✓' if native_count == 0 else f'✗ UNEXPECTED — found {native_count} native result(s) (should be 0)'
        print(f"  {sc['common_name']:30s} expect=absent    native_results={native_count}  introduced_results={introduced_count}  {status}")
    else:
        # Check monthly histogram for expected peak months
        r_hist = session.get(
            HISTOGRAM_URL,
            params={**_base, 'taxon_id': sc['taxon_id'], 'interval': 'month_of_year'},
        )
        r_hist.raise_for_status()
        monthly = r_hist.json().get('results', {}).get('month_of_year', {})
        counts_by_month = {int(k): int(v) for k, v in monthly.items()}
        if counts_by_month:
            peak_month = max(counts_by_month, key=counts_by_month.get)
            top3 = sorted(counts_by_month, key=counts_by_month.get, reverse=True)[:3]
            overlap = sc['peak_months'] & set(top3)
            status = '✓' if overlap else f'✗ peak months {top3} — expected overlap with {sorted(sc["peak_months"])}'
        else:
            status = '✗ no histogram data returned'
        print(f"  {sc['common_name']:30s} expect=native    native_results={native_count}  peak_months={top3}  {status}")

print()
print('If any ✗ appears above, re-examine native=true behavior before using outreach charts.')

Spot-check results:

  Monarch                        expect=native    native_results=1  peak_months=[8, 9, 10]  ✓
  Eastern Tiger Swallowtail      expect=native    native_results=0  peak_months=[1, 2, 3]  ✗ peak months [1, 2, 3] — expected overlap with [5, 6, 8]
  European Honeybee              expect=absent    native_results=0  introduced_results=1  ✓

If any ✗ appears above, re-examine native=true behavior before using outreach charts.


In [5]:
# -- Weekly overview: raw counts + effort-normalized fraction --
import plotly.graph_objects as go
from plotly.subplots import make_subplots

_weeks       = list(range(1, 53))
_month_ticks = [1, 5, 9, 14, 18, 22, 27, 31, 35, 40, 44, 48]
_month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

_fg_colors = {'Lepidoptera': '#1f77b4', 'Anthophila': '#ff7f0e'}
_ls_dashes  = {'overall': 'solid', 'adult': 'dash', 'larva': 'dot', 'pupa': 'dashdot'}

weekly = hist_df[hist_df['interval'] == 'week_of_year'].copy()

fig = make_subplots(
    rows=2, cols=1,
    shared_xaxes=True,
    subplot_titles=(
        'Raw observation counts (native taxa only)',
        'Effort-normalized — fraction of all verifiable observations that week',
    ),
    vertical_spacing=0.1,
)

for (fg, lsb), grp in weekly.groupby(['focus_group', 'life_stage_bucket']):
    if fg == 'Anthophila' and lsb != 'overall':
        continue
    grp  = grp.set_index('bin').reindex(_weeks, fill_value=0)
    name = f'{fg} – {lsb}'
    line = dict(color=_fg_colors[fg], dash=_ls_dashes[lsb])

    fig.add_scatter(x=_weeks, y=grp['count'],    mode='lines', name=name,
                    line=line, legendgroup=name, row=1, col=1)
    fig.add_scatter(x=_weeks, y=grp['fraction'],  mode='lines', name=name,
                    line=line, legendgroup=name, showlegend=False, row=2, col=1)

fig.update_xaxes(tickvals=_month_ticks, ticktext=_month_names)
fig.update_yaxes(title_text='Observations', row=1, col=1)
fig.update_yaxes(title_text='Fraction of all obs', tickformat='.1%', row=2, col=1)
fig.update_layout(
    title='Weekly native phenology — VA Piedmont (complete dataset)',
    height=650,
)
fig.show()

In [6]:
# -- Top native taxa via species_counts, then monthly histograms per taxon --
_top_rows = []
for fg, taxon_id in [('Lepidoptera', LEPIDOPTERA_TAXON_ID), ('Anthophila', ANTHOPHILA_TAXON_ID)]:
    r = session.get(
        SPECIES_COUNTS_URL,
        params={**_base, 'taxon_id': taxon_id, 'native': 'true', 'per_page': TOP_N},
    )
    r.raise_for_status()
    for rec in r.json().get('results', []):
        t = rec.get('taxon', {})
        _top_rows.append({
            'focus_group': fg,
            'taxon_id':    t.get('id'),
            'taxon_name':  t.get('name'),
            'common_name': t.get('preferred_common_name'),
            'total_obs':   rec.get('count'),
        })

top_taxa_df = pd.DataFrame(_top_rows)
print(top_taxa_df[['focus_group', 'common_name', 'taxon_name', 'total_obs']].to_string(index=False))

# Monthly histogram per top taxon (1 call each, complete data)
_monthly_rows = []
for _, row in top_taxa_df.iterrows():
    r = session.get(
        HISTOGRAM_URL,
        params={**_base, 'taxon_id': int(row['taxon_id']), 'interval': 'month_of_year'},
    )
    r.raise_for_status()
    for month_key, count in r.json().get('results', {}).get('month_of_year', {}).items():
        _monthly_rows.append({
            'focus_group': row['focus_group'],
            'taxon_id':    row['taxon_id'],
            'taxon_name':  row['taxon_name'],
            'common_name': row['common_name'],
            'month':       int(month_key),
            'count':       int(count),
        })

monthly_taxa_df = pd.DataFrame(_monthly_rows)

focus_group               common_name          taxon_name  total_obs
Lepidoptera Eastern Tiger Swallowtail     Papilio glaucus       7375
Lepidoptera             Huron Skipper    Atalopedes huron       5545
Lepidoptera                   Monarch    Danaus plexippus       5279
Lepidoptera            Pearl Crescent    Phyciodes tharos       3479
Lepidoptera       Eastern Tailed-Blue     Cupido comyntas       3223
 Anthophila Common Eastern Bumble Bee    Bombus impatiens       5888
 Anthophila     Eastern Carpenter Bee  Xylocopa virginica       3895
 Anthophila   Brown-belted Bumble Bee Bombus griseocollis       2190
 Anthophila    Two-spotted Bumble Bee  Bombus bimaculatus       1726
 Anthophila      Pure Green Sweat bee     Augochlora pura       1201


In [7]:
# -- Monthly outreach chart: top-N native taxa per focus group --
_month_order = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun',
                'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']

monthly_taxa_df['month_name'] = monthly_taxa_df['month'].apply(lambda m: _month_order[m - 1])
monthly_taxa_df['month_name'] = pd.Categorical(
    monthly_taxa_df['month_name'], categories=_month_order, ordered=True
)
monthly_taxa_df['label'] = monthly_taxa_df['common_name'].fillna(monthly_taxa_df['taxon_name'])

fig = px.bar(
    monthly_taxa_df.sort_values('month'),
    x='month_name',
    y='count',
    color='label',
    facet_row='focus_group',
    barmode='group',
    title=f'Monthly prevalence — top {TOP_N} native taxa per group (VA Piedmont, complete dataset)',
    labels={'count': 'Observations', 'month_name': '', 'label': 'Taxon'},
    category_orders={'month_name': _month_order},
)
fig.update_yaxes(matches=None)
fig.show()

In [8]:
# -- Life stage annotation coverage report --
# Low rates mean sparse volunteer annotation, not low biological prevalence.
# Use this to caveat any life-stage-specific claims in outreach materials.

_lep_overall = hist_df[
    (hist_df['focus_group'] == 'Lepidoptera') &
    (hist_df['life_stage_bucket'] == 'overall') &
    (hist_df['interval'] == 'week_of_year')
]['count'].sum()

print(f'Lepidoptera — native obs in Piedmont (all-time): {_lep_overall:,}')
print()
print('Life stage annotation coverage (annotated native obs / overall native obs):')
for lsb in ('adult', 'larva', 'pupa'):
    _n = hist_df[
        (hist_df['focus_group'] == 'Lepidoptera') &
        (hist_df['life_stage_bucket'] == lsb) &
        (hist_df['interval'] == 'week_of_year')
    ]['count'].sum()
    _pct = _n / _lep_overall if _lep_overall else float('nan')
    _flag = '  ⚠ sparse — interpret with caution' if _pct < 0.10 else ''
    print(f'  {lsb:6s}: {_n:>7,}  ({_pct:.1%}){_flag}')

Lepidoptera — native obs in Piedmont (all-time): 164,120

Life stage annotation coverage (annotated native obs / overall native obs):
  adult : 110,039  (67.0%)
  larva :  22,620  (13.8%)
  pupa  :     749  (0.5%)  ⚠ sparse — interpret with caution
